## 1. Carga y limpieza de los datos


In [3]:
import pandas as pd
import os

train = pd.read_csv("train_sent_emo.csv", encoding="utf-8")
dev = pd.read_csv("dev_sent_emo.csv", encoding="utf-8")
test = pd.read_csv("test_sent_emo.csv", encoding="utf-8")

def clean_text(s: str):
    if not isinstance(s, str):
        return s
    s = s.replace('\x92', "")
    s = "".join(ch for ch in s if ch.isprintable())
    return s

for df_ in(train, dev, test):
    df_["Utterance"] = df_["Utterance"].apply(clean_text)
train.head(5)
dev.head(5)
test.head(5)

,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,1,Why do all youre coffee mugs have numbers on t...,Mark,surprise,positive,0,0,3,19,"00:14:38,127","00:14:40,378"
1,2,Oh. Thats so Monica can keep track. That way i...,Rachel,anger,negative,0,1,3,19,"00:14:40,629","00:14:47,385"
2,3,Y'know what?,Rachel,neutral,neutral,0,2,3,19,"00:14:56,353","00:14:57,520"
3,19,"Come on, Lydia, you can do it.",Joey,neutral,neutral,1,0,1,23,"0:10:44,769","0:10:46,146"
4,20,Push!,Joey,joy,positive,1,1,1,23,"0:10:46,146","0:10:46,833"


# 2.LabelEncoder: convierte etiquetas de texto a números

In [4]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
train["label"] = label_encoder.fit_transform(train["Sentiment"])
dev["label"]   = label_encoder.transform(dev["Sentiment"])
test["label"]  = label_encoder.transform(test["Sentiment"])

num_labels = len(label_encoder.classes_)
print("Clases de sentimiento:", label_encoder.classes_)
print("Número de clases:", num_labels)

Clases de sentimiento: ['negative' 'neutral' 'positive']
Número de clases: 3


## 3. Carga del tokenizador y del modelo BERT




In [11]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4. Construcción de entradas con y sin contexto


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

#(1)Construcción de inputs con/sin contexto
CONTEXT_K = 3
MAX_LEN = 128
batch_size = 16
def _has_cols(df, cols):
    return all(c in df.columns for c in cols)
def build_text_inputs(df: pd.DataFrame, use_context: bool, k: int = 3, add_speaker: bool = True):
    """
    Devuelve lista de strings listos para tokenizar.
    - Si existen columnas típicas de MELD (Dialogue_ID y Utterance_ID), construye contexto por diálogo.
    - Si no, usa el orden actual del dataframe como secuencia (fallback).
    """
    if not use_context:
        if add_speaker and "Speaker" in df.columns:
            return ("[SPEAKER=" + df["Speaker"].astype(str) + "] " + df["Utterance"].astype(str)).tolist()
        return df["Utterance"].astype(str).tolist()

    #(2)Con contexto: caso estándar MELD
    if _has_cols(df, ["Dialogue_ID", "Utterance_ID"]):
        df2 = df.copy()
        df2["Utterance"] = df2["Utterance"].astype(str)
        if "Speaker" in df2.columns:
            df2["Speaker"] = df2["Speaker"].astype(str)
    df2 = df2.sort_values(["Dialogue_ID", "Utterance_ID"], kind="stable")
    inputs = []
    for _, g in df2.groupby("Dialogue_ID", sort=False):
            utts = g["Utterance"].tolist()
            spks = g["Speaker"].tolist() if "Speaker" in g.columns else [None] * len(g)
            for i in range(len(g)):
                start = max(0, i - k)
                parts = []
                for j in range(start, i + 1):
                    if add_speaker and spks[j] is not None:
                        parts.append(f"[SPEAKER={spks[j]}] {utts[j]}")
                    else:
                        parts.append(utts[j])
                inputs.append(" [SEP] ".join(parts))
    return inputs
    #(3)Con contexto: fallback (sin columnas MELD)
    utts = df["Utterance"].astype(str).tolist()
    spks = df["Speaker"].astype(str).tolist() if "Speaker" in df.columns else [None] * len(df)
    inputs = []
    for i in range(len(df)):
        start = max(0, i - k)
        parts = []
        for j in range(start, i + 1):
            if add_speaker and spks[j] is not None:
                parts.append(f"[SPEAKER={spks[j]}] {utts[j]}")
            else:
                parts.append(utts[j])
        inputs.append(" [SEP] ".join(parts))
    return inputs

Device: cuda


## 5. Preparación de los datos para PyTorch



In [13]:
from torch.utils.data import Dataset, DataLoader

class TextClsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len: int):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }
        return item

## 6. Entrenamiento y evaluación del modelo


In [17]:
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
def make_class_weights(labels, num_labels):
    counts = np.bincount(np.array(labels, dtype=int), minlength=num_labels).astype(np.float32)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / counts
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=device)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / max(1, len(loader))

@torch.no_grad()
@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    all_labels, all_preds = [], []
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(
        all_labels, all_preds,
        target_names=label_encoder.classes_,
        digits=4
    )
    return acc, f1, cm, report, np.array(all_preds)

## 7. Subconjunto fijo del conjunto de test




In [18]:
TEST_SAMPLE_SIZE = 500
rng = np.random.default_rng(SEED)
test_indices = np.arange(len(test))
if len(test_indices) > TEST_SAMPLE_SIZE:
    test_sample_indices = rng.choice(test_indices, size=TEST_SAMPLE_SIZE, replace=False)
else:
    test_sample_indices = test_indices
test_sample_indices = np.sort(test_sample_indices).tolist()

## 8. Definición de los tres enfoques BERT



In [19]:
def make_model():
    m = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    return m.to(device)

def run_experiment(name: str, use_context: bool, use_balanced: bool):
    # Textos
    texts_train = build_text_inputs(train, use_context=use_context, k=CONTEXT_K, add_speaker=True)
    texts_dev   = build_text_inputs(dev,   use_context=use_context, k=CONTEXT_K, add_speaker=True)
    texts_test_full = build_text_inputs(test, use_context=use_context, k=CONTEXT_K, add_speaker=True)

    labels_train = train["label"].tolist()
    labels_dev   = dev["label"].tolist()
    labels_test  = test["label"].tolist()

    # Subset fijo del test (fair sampling)
    texts_test_sub  = [texts_test_full[i] for i in test_sample_indices]
    labels_test_sub = [labels_test[i] for i in test_sample_indices]

    # DataLoaders
    train_ds = TextClsDataset(texts_train, labels_train, tokenizer, MAX_LEN)
    dev_ds   = TextClsDataset(texts_dev,   labels_dev,   tokenizer, MAX_LEN)
    test_ds  = TextClsDataset(texts_test_sub, labels_test_sub, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    dev_loader   = DataLoader(dev_ds,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

    # Modelo + optimizador
    model = make_model()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Loss (balanced o normal)
    if use_balanced:
        w = make_class_weights(labels_train, num_labels)
        criterion = nn.CrossEntropyLoss(weight=w)
    else:
        criterion = nn.CrossEntropyLoss()

    # Train con selección por dev macro-F1
    num_epochs = 2
    best_dev_f1 = -1.0
    best_state = None

    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        dev_acc, dev_f1, _, _, _ = eval_model(model, dev_loader)
        print(f"[{name}] Epoch {epoch+1}/{num_epochs} | train_loss={tr_loss:.4f} | dev_acc={dev_acc:.4f} | dev_f1={dev_f1:.4f}")

        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)
        model.to(device)

    # Test subset final
    test_acc, test_f1, cm, report, pred_ids = eval_model(model, test_loader)

    return {
        "name": name,
        "use_context": use_context,
        "balanced": use_balanced,
        "test_acc_sub": test_acc,
        "test_f1_sub": test_f1,
        "cm_sub": cm,
        "report_sub": report,
        "pred_ids_sub": pred_ids,
    }

# Ejecuta los 3 settings
results = []
results.append(run_experiment("BERT-Utt", use_context=False, use_balanced=False))
results.append(run_experiment("BERT-Context", use_context=True, use_balanced=False))
results.append(run_experiment("BERT-Context+Balanced", use_context=True, use_balanced=True))

summary = pd.DataFrame([{
    "Método": r["name"],
    "Contexto": r["use_context"],
    "Balanced": r["balanced"],
    "Acc(test_sub)": r["test_acc_sub"],
    "MacroF1(test_sub)": r["test_f1_sub"],
} for r in results])

summary

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[BERT-Utt] Epoch 1/2 | train_loss=0.7998 | dev_acc=0.6745 | dev_f1=0.6485
[BERT-Utt] Epoch 2/2 | train_loss=0.6437 | dev_acc=0.6745 | dev_f1=0.6458


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[BERT-Context] Epoch 1/2 | train_loss=0.8049 | dev_acc=0.6790 | dev_f1=0.6312
[BERT-Context] Epoch 2/2 | train_loss=0.6222 | dev_acc=0.6952 | dev_f1=0.6699


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[BERT-Context+Balanced] Epoch 1/2 | train_loss=0.8487 | dev_acc=0.6573 | dev_f1=0.6459
[BERT-Context+Balanced] Epoch 2/2 | train_loss=0.6566 | dev_acc=0.6646 | dev_f1=0.6498


,Método,Contexto,Balanced,Acc(test_sub),MacroF1(test_sub)
0,BERT-Utt,False,False,0.682,0.640613
1,BERT-Context,True,False,0.690,0.663910
2,BERT-Context+Balanced,True,True,0.650,0.639778


## 9. Comparación de resultados


In [20]:
import numpy as np
import pandas as pd

summary = pd.DataFrame([{
    "Método": r["name"],
    "Contexto": r["use_context"],
    "Balanced": r["balanced"],
    "Acc(test_sub)": r["test_acc_sub"],
    "MacroF1(test_sub)": r["test_f1_sub"],
} for r in results]).sort_values(["MacroF1(test_sub)", "Acc(test_sub)"], ascending=False)
display(summary)

best_method = summary.iloc[0]["Método"]
print("Best method:", best_method)

best_run = next(r for r in results if r["name"] == best_method)

print("\n=== Confusion Matrix (test_sub) ===")
print(best_run["cm_sub"])

print("\n=== Classification Report (test_sub) ===")
print(best_run["report_sub"])

,Método,Contexto,Balanced,Acc(test_sub),MacroF1(test_sub)
1,BERT-Context,True,False,0.690,0.663910
0,BERT-Utt,False,False,0.682,0.640613
2,BERT-Context+Balanced,True,True,0.650,0.639778


Best method: BERT-Context

=== Confusion Matrix (test_sub) ===
[[ 82  43  25]
 [ 42 199  16]
 [ 11  18  64]]

=== Classification Report (test_sub) ===
              precision    recall  f1-score   support

    negative     0.6074    0.5467    0.5754       150
     neutral     0.7654    0.7743    0.7698       257
    positive     0.6095    0.6882    0.6465        93

    accuracy                         0.6900       500
   macro avg     0.6608    0.6697    0.6639       500
weighted avg     0.6890    0.6900    0.6886       500



## 10. Análisis cualitativo de predicciones



In [23]:
# Subconjunto fijo del test (ya se generó antes con test_sample_indices)
test_sub = test.iloc[test_sample_indices].copy().reset_index(drop=True)

# Añadir las predicciones del mejor método
pred_ids = best_run["pred_ids_sub"].astype(int)
test_sub["pred_sentiment"] = label_encoder.inverse_transform(pred_ids)

# Columnas a mostrar (mantener la información de diálogo si existe)
cols = []
for c in ["Dialogue_ID", "Utterance_ID", "Speaker", "Utterance", "Sentiment", "pred_sentiment"]:
    if c in test_sub.columns:
        cols.append(c)

# (A) 10 ejemplos aleatorios
rng_show = np.random.default_rng(SEED)
rand_idx = rng_show.choice(np.arange(len(test_sub)), size=min(10, len(test_sub)), replace=False)
print("\n=== (A) 10 ejemplos aleatorios (Gold vs Pred) ===")
display(test_sub.loc[np.sort(rand_idx), cols])

# (B) 10 ejemplos con error (Gold != Pred)
err = test_sub[test_sub["Sentiment"] != test_sub["pred_sentiment"]].copy()
print("\n=== (B) 10 ejemplos con error (Gold != Pred) ===")
if len(err) == 0:
    print("No hay ejemplos con error (muy poco probable).")
else:
    display(err.head(10)[cols])

# (C) Ejemplos consecutivos del mismo Speaker (para analizar el diálogo)
print("\n=== (C) Ejemplos consecutivos del mismo Speaker ===")
if "Speaker" in test_sub.columns:
    top_speaker = test_sub["Speaker"].value_counts().index[0]
    print("Speaker:", top_speaker)
    display(test_sub[test_sub["Speaker"] == top_speaker].head(10)[cols])
else:
    print("No existe la columna Speaker, se omite esta parte.")


=== (A) 10 ejemplos aleatorios (Gold vs Pred) ===


,Dialogue_ID,Utterance_ID,Speaker,Utterance,Sentiment,pred_sentiment
42,25,15,Rachel,Yeah. Oh! Was how you invented the cotton gin?!,neutral,positive
43,28,6,Phoebe,Its kind of embarrassing.,neutral,negative
47,30,4,Rachel,Ross's what?,positive,neutral
100,64,1,Ross,"Well, I was going to stick it in the ATM, but ...",neutral,neutral
214,123,2,Monica,No! No-no! He is totally incompetent. I called...,negative,positive
216,123,7,Phoebe,He,neutral,neutral
322,178,3,Chandler,"Little baby girl Chandler, where I have heard ...",neutral,positive
347,193,4,Ross,"Okay, umm, why dont, why dont you all start t...",neutral,neutral
380,211,3,Phoebe,Joey!,negative,negative
425,246,8,Tour Guide,I shared my puddin which you man!,negative,positive



=== (B) 10 ejemplos con error (Gold != Pred) ===


,Dialogue_ID,Utterance_ID,Speaker,Utterance,Sentiment,pred_sentiment
0,2,2,Ross,"Uhh, yes I did but there isn't. Okay, here we go.",negative,neutral
1,2,5,Ross,"Oh okay, lift it straight up over your head!",negative,positive
4,5,7,Chandler,"What a coincidence, I listen in my sleep.",neutral,negative
7,7,2,Joey,"And its not fake, its totally brutal.",neutral,negative
8,7,4,Monica,"So you can like, bite, and pull peoples hair a...",negative,neutral
10,9,3,Phoebe,Oh!,negative,positive
16,12,5,Ross,Okay-okay!,negative,positive
20,15,4,Joey,Bye!,neutral,positive
29,21,4,Phoebe,"no, um, I'm sorry, don't tell me, I don't thin...",negative,neutral
32,22,2,Chandler,I just came over to drop offnothing. So that w...,neutral,negative



=== (C) Ejemplos consecutivos del mismo Speaker ===
Speaker: Chandler


,Dialogue_ID,Utterance_ID,Speaker,Utterance,Sentiment,pred_sentiment
4,5,7,Chandler,"What a coincidence, I listen in my sleep.",neutral,negative
9,8,4,Chandler,"Well, maybe he had some kind of uh, new, cool ...",neutral,neutral
19,14,10,Chandler,"Oh, all right.",neutral,neutral
32,22,2,Chandler,I just came over to drop offnothing. So that w...,neutral,negative
35,22,6,Chandler,"Well, y'know, you and me, it had to end sometime.",negative,neutral
45,29,6,Chandler,"Look, we have enough, just walk away.",negative,negative
83,47,9,Chandler,Hey!,neutral,positive
84,47,12,Chandler,"Oh yeah? Well, howd she take it?",neutral,neutral
88,49,9,Chandler,Hey!,positive,positive
89,49,13,Chandler,"Yeah, I can say that I have a conference and y...",neutral,neutral


In [22]:

os.makedirs("output", exist_ok=True)

out_path = "output/test_sub_500_best_bert_predictions.csv"
test_sub[cols].to_csv(out_path, index=False, encoding="utf-8")

print("Saved:", out_path)
print("Best method used:", best_method)

Saved: output/test_sub_500_best_bert_predictions.csv
Best method used: BERT-Context
